# Music Annotation Project V8.0e
## Read-Only Truth-Uncertainty, Registration-Sensitivity and Exact-Pixel Overlay Audit

**Runtime:** CPU is sufficient. Select **Runtime → Run all**.

V8.0e is a read-only forensic audit of the twice-reproduced V8.0d lock. It does **not** train, recalibrate, move a threshold, change a label, rescore the CNN, alter V7.8, use independent data, or authorise a freeze.

### Questions

1. Which final V8.0d PRINT pixels have registered truth `UNCERTAIN`?
2. What are their exact `(page, y, x)` coordinates?
3. Is clean printed ink present at the exact registered coordinate?
4. Would the pixel remain PRINT under all possible ±1-pixel or ±2-pixel registration perturbations?
5. Is the local clean-ink neighbourhood uniformly print, uniformly non-print, or mixed?
6. Where are the exact selected pixels in observed, clean-truth and V7.8-baseline crops?

### Interpretation boundary

The registered V8.0d truth definition remains unchanged. Alternative tests below are **post-result sensitivity diagnostics** only. They cannot be used to relabel training data, tune the model, change the PRINT rule or convert V8.0d into a development pass.

### Frozen sensitivity definitions

For the frozen clean-ink map:

- **Exact-centre evidence:** clean ink exists at `(y, x)`.
- **±1 shift-invariant PRINT:** every location in the 3×3 neighbourhood is clean ink.
- **±2 shift-invariant PRINT:** every location in the 5×5 neighbourhood is clean ink.
- **±1 shift-invariant NON-PRINT:** no location in the 3×3 neighbourhood is clean ink.
- **±2 shift-invariant NON-PRINT:** no location in the 5×5 neighbourhood is clean ink.
- Anything between the all/none cases is **mixed/unresolved**.

These are deliberately conservative perturbation-invariance tests. A pixel cannot be called registration-stable PRINT merely because one favourable nearby coordinate contains clean ink.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow matplotlib pillow


## 1. Frozen configuration and output directory


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, sys, platform
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V8D_ROOT=ROOT/'observed_print_classifier_v8_0d'
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_0e'
V8D_RUN_OVERRIDE=None
V75_RUN_OVERRIDE=None

EXPECTED_RUN_LOCK_SHA256='66be107973e6b8ea8716e0d11e48116952e8e5fdf85fd2ea9292ce4b5f52ea61'
EXPECTED_MANIFEST_SHA256='8ad9490658edcf898da7b04b2820d858a12bf4ec2ce4ab3a722d762a2831bcd7'
EXPECTED_FOLD_MAP_SHA256='ae1ac4557c1ffab1371094f3fe1c818fd0f652bb271512830ee06da55fc4867b'
EXPECTED_ALL_PRINT=124
EXPECTED_STABLE_PRINT=75
EXPECTED_STABLE_ANNOTATION=0
EXPECTED_TRUTH_UNCERTAIN_PRINT=49
CROP_RADIUS=48
ITEMS_PER_SHEET=12

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_0e_read_only_uncertainty_audit'
OUT=OUTPUT_ROOT/RUN
TABLES=OUT/'tables'; AUDIT=OUT/'audit'; OVERLAYS=OUT/'overlays'
for p in [OUT,TABLES,AUDIT,OVERLAYS]: p.mkdir(parents=True,exist_ok=False)
print('Output:',OUT)


## 2. Integrity helpers and twice-reproduced V8.0d lock gate


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()

def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def image_map(folder):
    folder=Path(folder)
    return {p.stem:p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if folder.exists() else {}

required_v8d=['experiment_manifest.json','audit/replay_test.json']
V8D=Path(V8D_RUN_OVERRIDE) if V8D_RUN_OVERRIDE else latest_complete(V8D_ROOT,required_v8d)
if V8D is None: raise RuntimeError('No complete V8.0d run found.')
manifest=json.loads((V8D/'experiment_manifest.json').read_text())
replay=json.loads((V8D/'audit'/'replay_test.json').read_text())
checks={
    'run_lock':manifest.get('run_lock_sha256')==EXPECTED_RUN_LOCK_SHA256==replay.get('run_lock_sha256'),
    'manifest_hash':manifest.get('manifest_sha256')==EXPECTED_MANIFEST_SHA256,
    'fold_hash':manifest.get('fold_map_sha256')==EXPECTED_FOLD_MAP_SHA256,
    'same_runtime_replay':replay.get('checks',{}).get('same_runtime_exact_replay_pass') is True,
    'previous_lock_found':replay.get('previous_completed_lock_found') is True,
    'previous_clean_run_match':replay.get('previous_clean_run_match') is True,
    'development_only':manifest.get('development_only') is True,
    'independent_not_used':manifest.get('independent_data_used') is False,
}
(AUDIT/'v8d_lock_checks.json').write_text(json.dumps(checks,indent=2))
if not all(checks.values()): raise RuntimeError(f'V8.0d lock/replay gate failed: {checks}')

pred_parquet=V8D/'tables'/'all_eligible_oof_predictions.parquet'
pred_csv=V8D/'tables'/'all_eligible_oof_predictions.csv'
if pred_parquet.exists():
    pred_path=pred_parquet; predictions=pd.read_parquet(pred_path)
elif pred_csv.exists():
    pred_path=pred_csv; predictions=pd.read_csv(pred_path)
else:
    raise RuntimeError('Missing all_eligible_oof_predictions.parquet/.csv in locked V8.0d run.')
required_cols={'page','source_group','y','x','truth','candidate_id','fold','raw_print','p_print','decision_pre_run_rule','decision'}
missing=required_cols-set(predictions.columns)
if missing: raise RuntimeError(f'Prediction table missing columns: {sorted(missing)}')
if len(predictions)!=manifest.get('all_eligible_rows'): raise RuntimeError('Prediction row count differs from V8.0d manifest.')
print('Locked V8.0d:',V8D)
print('Prediction input:',pred_path)
print('Rows:',len(predictions))
print(json.dumps(checks,indent=2))


## 3. Reconcile all final PRINT decisions before inspecting uncertainty


In [ ]:
selected=predictions[predictions.decision.eq('PRINT')].copy()
selected['truth_name']=selected.truth.map({0:'ANNOTATION',1:'PRINT',2:'UNCERTAIN'})
counts=selected.truth_name.value_counts().to_dict()
reconciliation={
    'all_final_print':int(len(selected)),
    'stable_print':int((selected.truth==1).sum()),
    'stable_annotation':int((selected.truth==0).sum()),
    'truth_uncertain':int((selected.truth==2).sum()),
}
expected={'all_final_print':EXPECTED_ALL_PRINT,'stable_print':EXPECTED_STABLE_PRINT,
          'stable_annotation':EXPECTED_STABLE_ANNOTATION,'truth_uncertain':EXPECTED_TRUTH_UNCERTAIN_PRINT}
if reconciliation!=expected: raise RuntimeError(f'Locked-result reconciliation mismatch: {reconciliation} != {expected}')
selected.to_csv(TABLES/'all_final_print_pixels.csv',index=False)
uncertain=selected[selected.truth.eq(2)].copy().sort_values(['page','y','x']).reset_index(drop=True)
uncertain['audit_pixel_id']=np.arange(1,len(uncertain)+1)
print(json.dumps(reconciliation,indent=2))


## 4. Load frozen clean, observed and V7.8 baseline arrays


In [ ]:
clean_map=image_map(ROOT/'source'/'clean')
observed_map=image_map(ROOT/'aligned')
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest_complete(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: raise RuntimeError('Complete V7.5 run not found.')
status=pd.read_csv(V75/'tables'/'cache_status.csv')
cache_map={str(r.page):Path(r.cache) for _,r in status.iterrows()}
needed=sorted(selected.page.unique())
assets={}; asset_audit=[]
for page in needed:
    if page not in clean_map or page not in observed_map or page not in cache_map:
        raise RuntimeError(f'{page}: missing clean, observed or V7.5 cache input.')
    clean=cv2.imread(str(clean_map[page]),0); observed=cv2.imread(str(observed_map[page]),0)
    with np.load(cache_map[page]) as z:
        if 'mask' not in z.files or 'geometry_t80' not in z.files: raise RuntimeError(f'{page}: cache lacks mask/geometry_t80')
        mask=z['mask']>0; baseline=z['geometry_t80']
    if clean is None or observed is None or clean.shape!=observed.shape or mask.shape!=observed.shape or baseline.shape!=observed.shape:
        raise RuntimeError(f'{page}: read/shape mismatch')
    assets[page]={'clean':clean,'observed':observed,'mask':mask,'baseline':baseline}
    asset_audit.append({'page':page,'clean_sha256':sha256_file(clean_map[page]),
                        'observed_sha256':sha256_file(observed_map[page]),
                        'v75_cache_sha256':sha256_file(cache_map[page]),'height':observed.shape[0],'width':observed.shape[1]})
pd.DataFrame(asset_audit).to_csv(AUDIT/'asset_hashes.csv',index=False)
print('Loaded assets for',len(assets),'selected pages from V7.5:',V75)


## 5. Recreate the frozen clean-ink map and audit perturbation invariance

`strict_class_r1` and `strict_class_r2` are sensitivity results, not replacement labels.


In [ ]:
def printed_ink(clean):
    return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0

def neighbourhood(a,y,x,r):
    h,w=a.shape
    y0=max(0,y-r); y1=min(h,y+r+1); x0=max(0,x-r); x1=min(w,x+r+1)
    crop=a[y0:y1,x0:x1]
    expected=(2*r+1)**2
    if crop.size!=expected: return None
    return crop

def strict_class(crop):
    if crop is None: return 'BORDER_UNRESOLVED'
    if bool(crop.all()): return 'PRINT_SHIFT_INVARIANT'
    if not bool(crop.any()): return 'NONPRINT_SHIFT_INVARIANT'
    return 'MIXED_UNRESOLVED'

ink_maps={page:printed_ink(z['clean']) for page,z in assets.items()}
records=[]
for _,r in uncertain.iterrows():
    page=str(r.page); y=int(r.y); x=int(r.x); ink=ink_maps[page]
    n1=neighbourhood(ink,y,x,1); n2=neighbourhood(ink,y,x,2)
    exact=bool(ink[y,x])
    rec=r.to_dict()
    rec.update({
        'exact_centre_clean_ink':exact,
        'ink_count_3x3':int(n1.sum()) if n1 is not None else np.nan,
        'ink_fraction_3x3':float(n1.mean()) if n1 is not None else np.nan,
        'strict_class_r1':strict_class(n1),
        'print_shift_invariant_r1':bool(n1 is not None and n1.all()),
        'nonprint_shift_invariant_r1':bool(n1 is not None and not n1.any()),
        'ink_count_5x5':int(n2.sum()) if n2 is not None else np.nan,
        'ink_fraction_5x5':float(n2.mean()) if n2 is not None else np.nan,
        'strict_class_r2':strict_class(n2),
        'print_shift_invariant_r2':bool(n2 is not None and n2.all()),
        'nonprint_shift_invariant_r2':bool(n2 is not None and not n2.any()),
    })
    records.append(rec)
audit=pd.DataFrame(records)

# Recreate the registered one-pixel boundary class and prove every input row remains registered UNCERTAIN.
for page,g in audit.groupby('page'):
    ink=ink_maps[page]; k=np.ones((3,3),np.uint8)
    near=cv2.dilate(ink.astype(np.uint8),k)>0
    interior=cv2.erode(ink.astype(np.uint8),k)>0
    boundary=np.logical_xor(near,interior)
    audit.loc[g.index,'registered_boundary_recreated']=[bool(boundary[int(y),int(x)]) for y,x in zip(g.y,g.x)]
if not audit.registered_boundary_recreated.all(): raise RuntimeError('At least one truth-UNCERTAIN/PRINT row did not recreate as the registered boundary class.')

# Map exact authoritative-mask component IDs.
for page,g in audit.groupby('page'):
    n,labels,stats,_=cv2.connectedComponentsWithStats(assets[page]['mask'].astype(np.uint8),8)
    audit.loc[g.index,'component_id']=[int(labels[int(y),int(x)]) for y,x in zip(g.y,g.x)]
audit['component_id']=audit.component_id.astype(int)
audit.to_csv(TABLES/'truth_uncertain_final_print_pixel_audit.csv',index=False)
print('Exact audited pixels:',len(audit))
display(audit[['audit_pixel_id','page','y','x','component_id','exact_centre_clean_ink','strict_class_r1','strict_class_r2','ink_fraction_3x3','ink_fraction_5x5']])


## 6. Summaries by audit class, page and component


In [ ]:
summary={
    'locked_v8d_run_lock_sha256':EXPECTED_RUN_LOCK_SHA256,
    'final_print_total':int(len(selected)),
    'truth_uncertain_final_print':int(len(audit)),
    'exact_centre_clean_ink':int(audit.exact_centre_clean_ink.sum()),
    'exact_centre_no_clean_ink':int((~audit.exact_centre_clean_ink).sum()),
    'print_shift_invariant_r1':int(audit.print_shift_invariant_r1.sum()),
    'nonprint_shift_invariant_r1':int(audit.nonprint_shift_invariant_r1.sum()),
    'mixed_or_border_r1':int((~audit.print_shift_invariant_r1 & ~audit.nonprint_shift_invariant_r1).sum()),
    'print_shift_invariant_r2':int(audit.print_shift_invariant_r2.sum()),
    'nonprint_shift_invariant_r2':int(audit.nonprint_shift_invariant_r2.sum()),
    'mixed_or_border_r2':int((~audit.print_shift_invariant_r2 & ~audit.nonprint_shift_invariant_r2).sum()),
    'interpretation':'Post-result read-only sensitivity audit; registered V8.0d labels and decisions are unchanged.',
}
(OUT/'uncertainty_audit_summary.json').write_text(json.dumps(summary,indent=2))

page_summary=audit.groupby(['page','source_group'],as_index=False).agg(
    selected_truth_uncertain=('audit_pixel_id','count'),
    exact_centre_clean_ink=('exact_centre_clean_ink','sum'),
    print_shift_invariant_r1=('print_shift_invariant_r1','sum'),
    nonprint_shift_invariant_r1=('nonprint_shift_invariant_r1','sum'),
    print_shift_invariant_r2=('print_shift_invariant_r2','sum'),
    nonprint_shift_invariant_r2=('nonprint_shift_invariant_r2','sum'),
    mean_ink_fraction_3x3=('ink_fraction_3x3','mean'),
    mean_ink_fraction_5x5=('ink_fraction_5x5','mean'))
page_summary.to_csv(TABLES/'uncertainty_audit_by_page.csv',index=False)
component_summary=audit.groupby(['page','source_group','component_id'],as_index=False).agg(
    selected_truth_uncertain=('audit_pixel_id','count'),
    y_min=('y','min'),y_max=('y','max'),x_min=('x','min'),x_max=('x','max'),
    exact_centre_clean_ink=('exact_centre_clean_ink','sum'),
    print_shift_invariant_r1=('print_shift_invariant_r1','sum'),
    nonprint_shift_invariant_r1=('nonprint_shift_invariant_r1','sum'),
    print_shift_invariant_r2=('print_shift_invariant_r2','sum'),
    nonprint_shift_invariant_r2=('nonprint_shift_invariant_r2','sum'))
component_summary.to_csv(TABLES/'uncertainty_audit_by_component.csv',index=False)
print(json.dumps(summary,indent=2)); display(page_summary); display(component_summary)


## 7. Exact selected-pixel overlays by component

Marker key:

- **Amber X:** final V8.0d PRINT with registered truth UNCERTAIN.
- **Green ring on clean panel:** exact centre contains clean ink.
- **Red ring on clean panel:** exact centre contains no clean ink.
- Dashed cyan rectangle: 33×33 model footprint around the selected-pixel cluster.


In [ ]:
def crop_bounds(shape,ys,xs,r=CROP_RADIUS):
    h,w=shape; return max(0,int(min(ys))-r),min(h,int(max(ys))+r+1),max(0,int(min(xs))-r),min(w,int(max(xs))+r+1)

def rgb(gray): return cv2.cvtColor(gray,cv2.COLOR_GRAY2RGB)

def draw_marker(ax,x,y,colour,marker='x',size=75,lw=1.8):
    ax.scatter([x],[y],s=size,c=colour,marker=marker,linewidths=lw,zorder=5)

for (page,cid),g in audit.groupby(['page','component_id']):
    z=assets[page]; y0,y1,x0,x1=crop_bounds(z['observed'].shape,g.y,g.x)
    fig,axes=plt.subplots(1,3,figsize=(15,5))
    panels=[('Observed',z['observed']),('Clean truth',z['clean']),('V7.8 baseline',z['baseline'])]
    for ax,(title,image) in zip(axes,panels):
        ax.imshow(image[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255)
        ax.set_title(f'{page} component {cid} — {title}')
        ax.axis('off')
        # 33×33 envelope around cluster, clipped to crop coordinates.
        rx=max(0,int(g.x.min())-16-x0); ry=max(0,int(g.y.min())-16-y0)
        rw=min(x1-x0,int(g.x.max())+17-x0)-rx; rh=min(y1-y0,int(g.y.max())+17-y0)-ry
        ax.add_patch(plt.Rectangle((rx,ry),rw,rh,fill=False,edgecolor='cyan',linestyle='--',linewidth=1.0))
    for _,r in g.iterrows():
        px=int(r.x)-x0; py=int(r.y)-y0
        draw_marker(axes[0],px,py,'orange','x')
        ring='lime' if bool(r.exact_centre_clean_ink) else 'red'
        draw_marker(axes[1],px,py,ring,'o',90,1.8)
        draw_marker(axes[2],px,py,'orange','x')
    fig.suptitle(f'{len(g)} truth-UNCERTAIN final PRINT pixel(s); exact coordinates overlaid')
    plt.tight_layout()
    fig.savefig(OVERLAYS/f'{page}_component_{cid}_exact_overlay.png',dpi=180,bbox_inches='tight')
    plt.close(fig)
print('Component overlay files:',len(list(OVERLAYS.glob('*_exact_overlay.png'))))


## 8. Paginated per-pixel contact sheets

Each row shows one exact selected coordinate in observed, clean and V7.8 panels. The crop is centred on the selected pixel; no component-centre proxy is used.


In [ ]:
def fixed_crop(image,y,x,r=32):
    pad=np.pad(image,r,mode='reflect'); return pad[y:y+2*r+1,x:x+2*r+1]

rows=audit.sort_values(['page','y','x']).reset_index(drop=True)
per_page=ITEMS_PER_SHEET
for sheet,start in enumerate(range(0,len(rows),per_page),1):
    q=rows.iloc[start:start+per_page]
    fig,axes=plt.subplots(len(q),3,figsize=(10,3*len(q)),squeeze=False)
    for rr,(_,r) in enumerate(q.iterrows()):
        z=assets[r.page]; ims=[z['observed'],z['clean'],z['baseline']]
        titles=['Observed','Clean truth','V7.8 baseline']
        for cc,(im,title) in enumerate(zip(ims,titles)):
            ax=axes[rr,cc]; crop=fixed_crop(im,int(r.y),int(r.x)); ax.imshow(crop,cmap='gray',vmin=0,vmax=255)
            colour='lime' if cc==1 and bool(r.exact_centre_clean_ink) else ('red' if cc==1 else 'orange')
            marker='o' if cc==1 else 'x'; draw_marker(ax,32,32,colour,marker,95,1.8)
            ax.set_title(f'#{int(r.audit_pixel_id)} {r.page} ({int(r.y)},{int(r.x)}) — {title}',fontsize=8); ax.axis('off')
    plt.tight_layout(); fig.savefig(OVERLAYS/f'pixel_contact_sheet_{sheet:02d}.png',dpi=170,bbox_inches='tight'); plt.close(fig)
print('Pixel contact sheets:',len(list(OVERLAYS.glob('pixel_contact_sheet_*.png'))))


## 9. Fail-closed counterfactual and audit manifest

The counterfactual is descriptive: it reports what would remain if development truth-UNCERTAIN final PRINT pixels were treated as unresolved evidence. It does not alter V8.0d outputs.


In [ ]:
counterfactual={
    'original_final_print_pixels':int(len(selected)),
    'registered_stable_print_pixels':int((selected.truth==1).sum()),
    'registered_stable_annotation_pixels':int((selected.truth==0).sum()),
    'registered_truth_uncertain_pixels':int((selected.truth==2).sum()),
    'descriptive_if_truth_uncertain_retained_on_v78_path':int((selected.truth==1).sum()),
    'pixels_withheld_as_unresolved':int((selected.truth==2).sum()),
    'warning':'Development-truth counterfactual only; clean truth is prohibited at inference.'}
(OUT/'fail_closed_counterfactual.json').write_text(json.dumps(counterfactual,indent=2))

manifest_out={
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'version':'8.0e-read-only-uncertainty-registration-audit',
    'source_v8d_run':str(V8D),
    'source_v8d_run_lock_sha256':EXPECTED_RUN_LOCK_SHA256,
    'source_prediction_file':str(pred_path),
    'source_prediction_sha256':sha256_file(pred_path),
    'v8d_lock_checks':checks,
    'model_trained':False,'model_rescored':False,'calibration_changed':False,
    'threshold_changed':False,'registered_labels_changed':False,'v78_changed':False,
    'independent_data_used':False,'freeze_permitted':False,
    'sensitivity_definitions':{
        'exact_centre':'clean ink at exact registered coordinate',
        'r1_print':'all 3x3 possible coordinates contain clean ink',
        'r1_nonprint':'no 3x3 possible coordinate contains clean ink',
        'r2_print':'all 5x5 possible coordinates contain clean ink',
        'r2_nonprint':'no 5x5 possible coordinate contains clean ink'},
    'software':{'python':sys.version,'opencv':cv2.__version__,'numpy':np.__version__,
                'pandas':pd.__version__,'platform':platform.platform()}}
(OUT/'audit_manifest.json').write_text(json.dumps(manifest_out,indent=2))
print(json.dumps(counterfactual,indent=2))
print('Audit complete:',OUT)


## Outputs to share

Please share:

```text
uncertainty_audit_summary.json
fail_closed_counterfactual.json
audit_manifest.json

audit/
  v8d_lock_checks.json
  asset_hashes.csv

tables/
  truth_uncertain_final_print_pixel_audit.csv
  uncertainty_audit_by_page.csv
  uncertainty_audit_by_component.csv

overlays/
  *_exact_overlay.png
  pixel_contact_sheet_*.png
```

The most important table is `truth_uncertain_final_print_pixel_audit.csv`; it contains all 49 exact coordinates and every sensitivity result.
